# 🏆 Multi-Series Time Series Forecasting — Chronos-2 × LightGBM × CatBoost + OOF Stacking

**Competition Notebook** | *End-to-End Pipeline*

---

### 📌 Objective

This notebook tackles **multi-series time series forecasting** where each series is identified by a unique `(code, sub_code)` pair.  
Rather than treating it as a single regression problem, we build a **diverse ensemble pipeline**:

| Component | Role |
|---|---|
| **Chronos-2** (Foundation Model) | Extracts global pattern features via horizon-mean predictions |
| **LightGBM** | Fast gradient-boosted trees with native categorical support |
| **CatBoost** | Ordered boosting — strong on categorical-heavy data |
| **Ridge (Stacking)** | Linearly blends OOF predictions for the final output |

> **Key Idea:** Chronos-2 is **not** used for direct forecasting.  
> Instead, the **mean of its horizon predictions** is injected as a feature — capturing global temporal patterns that tree models alone cannot learn.

## 🗺️ Pipeline Overview

```
Raw Data (.parquet)
  │
  ├─▶ 1. Data Loading & Memory Optimization
  │
  ├─▶ 2. Feature Engineering
  │       ├── Lag features (shift to avoid leakage)
  │       ├── Rolling statistics (mean, std, ewma)
  │       └── Target encoding (computed per CV fold)
  │
  ├─▶ 3. Chronos-2 Feature Extraction
  │       └── Mean of horizon predictions → global pattern feature
  │
  ├─▶ 4. Time-Series Cross Validation
  │       ├── Temporal split (no future leakage)
  │       └── Multi-seed averaging per fold
  │
  ├─▶ 5. Model Training (LightGBM + CatBoost)
  │       └── OOF predictions collected per fold
  │
  ├─▶ 6. OOF Stacking (Ridge Regression)
  │       └── Linear blend of base model OOF outputs
  │
  └─▶ 7. Test Inference & Submission
```

---
## 1️⃣ Imports & Configuration

All dependencies are loaded upfront. We also define **global constants** to keep the pipeline configurable.

In [1]:
!pip install git+https://github.com/amazon-science/chronos-forecasting.git

import pandas as pd
import numpy as np
import torch
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor
import warnings
import os
warnings.filterwarnings('ignore')

# -- Global Config -------------------------------------------------
DATA_PATH   = '/kaggle/input/competitions/ts-forecasting'
TRAIN_FILE  = 'train.parquet'
TEST_FILE   = 'test.parquet'
ID_COL      = 'id'
PRED_COL    = 'prediction'

TARGET        = 'y_target'
TIME_COL      = 'ts_index'
HORIZON_COL   = 'horizon'
GROUP_COLS    = ['code', 'sub_code']
CAT_COLS      = ['sub_category', 'sub_code']
SUPPORTED_HORIZONS = [1, 3, 10, 25]

N_SPLITS     = 5
SEEDS        = [42, 69, 100]
CLIP_MIN     = 1e-6
CLIP_MAX     = 1000.0
SUBMISSION_PATH = '/kaggle/working/submission.csv'

# Chronos controls: only trusted for short horizon in this pipeline
CHRONOS_HORIZONS = [1]
CHRONOS_NUM_SAMPLES = 5
CHRONOS_CACHE_PATH = '/kaggle/working/chronos_features.parquet'
CHRONOS_NAN_THRESHOLD = 0.30
CHRONOS_CONST_STD = 1e-6

# Use a safe dtype for CPU-only runs
CHRONOS_DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

  Cloning https://github.com/amazon-science/chronos-forecasting.git to /tmp/pip-req-build-p69wp0in
  Running command git clone --filter=blob:none --quiet https://github.com/amazon-science/chronos-forecasting.git /tmp/pip-req-build-p69wp0in
  Resolved https://github.com/amazon-science/chronos-forecasting.git to commit 6d68ed7c4ed2805d122d77b4660765b4089de5ca
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.5 MB/s eta 0:00:00
  Created wheel for chronos-forecasting: filename=chronos_forecasting-2.2.2-py3-none-any.whl size=73971 sha256=50f7208b07bcbe9242daf8de71452d2118c898a4a80f43d3452490538c6b01d0
  Stored in directory: /tmp/pip-ephem-wheel-cache-oa843k2n/wheels/b9/a6/b5/75fca7306751a

---
## 2️⃣ Chronos-2 — Foundation Model Initialization

We load the **Chronos-T5-Small** pretrained checkpoint from Amazon Science.  
Weights are cached to `/kaggle/working/` so re-runs don't re-download.

| Parameter | Value | Why |
|---|---|---|
| `device_map` | `cuda` if available | GPU inference is ~10× faster |
| `torch_dtype` | `bfloat16` | Halves memory with minimal precision loss |
| `cache_dir` | `/kaggle/working/chronos-weights` | Persists across kernel restarts |

In [2]:
from chronos import ChronosPipeline

# Ensure model cache directory exists in Kaggle Workspace
model_cache_dir = '/kaggle/working/chronos-weights'
os.makedirs(model_cache_dir, exist_ok=True)

# Initialize Chronos-2 and cache weights
print("Loading Chronos-2 weights to:", model_cache_dir)
chronos = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map="cuda" if torch.cuda.is_available() else "cpu",
    torch_dtype=CHRONOS_DTYPE,
    cache_dir=model_cache_dir
)

2026-03-23 18:30:21.148548: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774290621.373403      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774290621.433132      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774290621.963706      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774290621.963744      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774290621.963747      23 computation_placer.cc:177] computation placer alr

Loading Chronos-2 weights to: /kaggle/working/chronos-weights


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/185M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

---
## 3️⃣ Utility — Memory Optimization

Kaggle kernels have limited RAM. This function **downcasts** numeric columns to their smallest viable dtype, typically reducing memory by **60–80%**.

In [3]:
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory before: {start_mem:.2f} MB')
    
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and col_type.name != 'category' and 'datetime' not in col_type.name:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
                    
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory after:  {end_mem:.2f} MB')
    print(f'Reduced by:    {100 * (start_mem - end_mem) / start_mem:.1f}%\n')
    
    return df

---
## 4️⃣ Data Loading

The dataset is stored as **Parquet** files — columnar format that is faster to load and more memory-efficient than CSV.

The data is a **multi-series** time series:

- Each series is uniquely identified by `(code, sub_code)`
- Time steps are indexed by `ts_index` (integer, not datetime)
- Target variable: `y_target`

> ⚠️ **Update the path** below to match your competition's dataset name.

In [4]:
# ── Load Training + Test Data ───────────────────────────────
train_path = os.path.join(DATA_PATH, TRAIN_FILE)
test_path = os.path.join(DATA_PATH, TEST_FILE)

def load_table(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext == ".csv":
        try:
            return pd.read_csv(path, encoding="utf-8")
        except UnicodeDecodeError:
            # Fallback for non-UTF8 CSVs (common in Kaggle datasets)
            return pd.read_csv(path, encoding="latin-1")
    raise ValueError(f"Unsupported file type: {ext}")

train = load_table(train_path)
test_df = load_table(test_path)

train = reduce_mem_usage(train)
test_df = reduce_mem_usage(test_df)

train[TIME_COL] = pd.to_numeric(train[TIME_COL])
test_df[TIME_COL] = pd.to_numeric(test_df[TIME_COL])

train = train.sort_values(GROUP_COLS + [TIME_COL]).reset_index(drop=True)
test_df = test_df.sort_values(GROUP_COLS + [TIME_COL]).reset_index(drop=True)

if ID_COL not in test_df.columns:
    test_df[ID_COL] = np.arange(len(test_df))

df_raw = train
print(f"Train shape: {df_raw.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"Series count: {df_raw.groupby(GROUP_COLS).ngroups}")

Memory before: 3766.71 MB
Memory after:  1242.00 MB
Reduced by:    67.0%

Memory before: 999.17 MB
Memory after:  325.70 MB
Reduced by:    67.4%

Train shape: (5337414, 94)
Test shape:  (1447107, 92)
Series count: 1856


---
## 5️⃣ Feature Engineering — Statistical Features

We create features **per series** (grouped by `code × sub_code`) to preserve temporal structure.

| Feature Type | Details | Why |
|---|---|---|
| **Lag features** | `shift(1, 3, 5, 10)` | Captures recent values; `shift()` ensures **no future leakage** |
| **Rolling mean** | Windows of `5, 10, 20` | Smooths noise, captures local trend |
| **Rolling std** | Windows of `5, 10, 20` | Measures local volatility |
| **EWMA** | Span = 10 | Exponentially-weighted mean — reacts faster to recent changes |

> 🔒 **Leakage Prevention:** All features use `shift()` or only look at past values within each group.  
> `min_periods=1` ensures early rows still get valid values instead of NaN.

In [5]:
def get_horizon_feature_config(h):
    if h == 1:
        return {
            'lags': [1, 2, 3],
            'rolls': [3, 5],
            'ewm_span': 5,
            'long_only': False,
        }
    if h == 3:
        return {
            'lags': [1, 2, 3, 4, 5, 6, 7],
            'rolls': [3, 5, 7],
            'ewm_span': 7,
            'long_only': False,
        }
    return {
        'lags': [7, 14, 21, 30],
        'rolls': [7, 14, 30],
        'ewm_span': 14,
        'long_only': True,
    }


def add_horizon_meta_features(df):
    df = df.copy()
    df['log_horizon'] = np.log1p(df[HORIZON_COL])
    df['is_short'] = (df[HORIZON_COL] <= 3).astype(np.int8)
    return df


def build_statistical_features(df, value_col=TARGET, history_df=None):
    df = df.sort_values(GROUP_COLS + [HORIZON_COL, TIME_COL]).copy()

    if value_col in df.columns:
        source_df = df.copy()
        reuse_stats = False
    else:
        if history_df is None or value_col not in history_df.columns:
            raise KeyError(f"Value column '{value_col}' not found. Provide history_df with that column.")
        source_df = history_df.sort_values(GROUP_COLS + [HORIZON_COL, TIME_COL]).copy()
        reuse_stats = True

    stats = pd.DataFrame(index=source_df.index)

    for h in sorted(source_df[HORIZON_COL].dropna().unique()):
        cfg = get_horizon_feature_config(int(h))
        idx = source_df.index[source_df[HORIZON_COL] == h]
        part = source_df.loc[idx]
        grp = part.groupby(GROUP_COLS)[value_col]

        for lag in cfg['lags']:
            stats.loc[idx, f'lag_{lag}'] = grp.shift(lag)

        for window in cfg['rolls']:
            shifted = grp.shift(1)
            stats.loc[idx, f'rolling_mean_{window}'] = shifted.groupby(part[GROUP_COLS].apply(tuple, axis=1)).transform(
                lambda x: x.rolling(window, min_periods=1).mean()
            )
            stats.loc[idx, f'rolling_std_{window}'] = shifted.groupby(part[GROUP_COLS].apply(tuple, axis=1)).transform(
                lambda x: x.rolling(window, min_periods=1).std(ddof=0)
            )

        stats.loc[idx, f'ewma_{cfg["ewm_span"]}'] = grp.shift(1).groupby(part[GROUP_COLS].apply(tuple, axis=1)).transform(
            lambda x: x.ewm(span=cfg['ewm_span'], adjust=False).mean()
        )

        if cfg['long_only']:
            rm_7 = grp.shift(1).groupby(part[GROUP_COLS].apply(tuple, axis=1)).transform(
                lambda x: x.rolling(7, min_periods=1).mean()
            )
            rm_30 = grp.shift(1).groupby(part[GROUP_COLS].apply(tuple, axis=1)).transform(
                lambda x: x.rolling(30, min_periods=1).mean()
            )
            stats.loc[idx, 'trend_7_30'] = rm_7 - rm_30

    if not reuse_stats:
        out = pd.concat([df, stats], axis=1)
        return add_horizon_meta_features(out).fillna(-1)

    stats_df = pd.concat([source_df[GROUP_COLS + [HORIZON_COL, TIME_COL]], stats], axis=1)
    last_stats = stats_df.groupby(GROUP_COLS + [HORIZON_COL]).tail(1).drop(columns=[TIME_COL])
    out = df.merge(last_stats, on=GROUP_COLS + [HORIZON_COL], how='left')
    return add_horizon_meta_features(out).fillna(-1)

---
## 6️⃣ Feature Engineering — Chronos-2 Global Pattern Features

Chronos is used as a **feature extractor**, not a direct forecaster.

### Reliability Constraint Applied
- Chronos features are generated and used only for horizon `h = 1`
- For horizons `3, 10, 25`, Chronos features are dropped from model training

### Quality Safeguards
- If Chronos feature has `>30%` NaN, missing values are filled with `-1`
- If a Chronos feature is nearly constant, it is removed automatically

This keeps useful short-horizon Chronos signal while avoiding noisy long-horizon collapse.

In [6]:
def sanitize_chronos_features(df, feature_cols, nan_threshold=CHRONOS_NAN_THRESHOLD, const_std=CHRONOS_CONST_STD):
    removed = []
    for col in feature_cols:
        if col not in df.columns:
            continue

        nan_ratio = df[col].isna().mean()
        if nan_ratio > nan_threshold:
            print(f"[Chronos] {col}: NaN ratio={nan_ratio:.2%} > {nan_threshold:.0%}; filling with -1")
            df[col] = df[col].fillna(-1)
        else:
            df[col] = df[col].fillna(df[col].median())

        std_val = df[col].std(ddof=0)
        nunique = df[col].nunique(dropna=True)
        if nunique <= 1 or (pd.notna(std_val) and std_val < const_std):
            removed.append(col)

    if removed:
        print(f"[Chronos] Removing near-constant features: {removed}")
        df = df.drop(columns=removed)

    return df, [c for c in feature_cols if c not in removed]


def generate_chronos_features_batch(df, target_col=TARGET, horizons=None, context_len=64, history_df=None, num_samples=10, cache_path=None):
    print("Extracting Chronos features...")
    df = df.sort_values(GROUP_COLS + [HORIZON_COL, TIME_COL]).copy()

    if horizons is None:
        horizons = [1]

    feature_cols = [f'chronos_pred_h{h}' for h in horizons]
    for col in feature_cols:
        df[col] = np.nan

    group_keys = GROUP_COLS + [HORIZON_COL]

    if target_col in df.columns:
        grouped = df.groupby(group_keys)

        def get_values(key):
            return grouped.get_group(key).sort_values(TIME_COL)[target_col].values
    else:
        if history_df is None:
            raise KeyError(f"Target column '{target_col}' not found. Provide history_df for feature extraction.")
        history_df = history_df.sort_values(group_keys + [TIME_COL])
        history_grouped = history_df.groupby(group_keys)
        grouped = df.groupby(group_keys)

        def get_values(key):
            if key in history_grouped.groups:
                return history_grouped.get_group(key).sort_values(TIME_COL)[target_col].values
            return np.array([0.0], dtype=np.float32)

    cached_groups = set()
    if cache_path and os.path.exists(cache_path):
        cache_df = pd.read_parquet(cache_path)
        expected_cols = group_keys + feature_cols
        if all(col in cache_df.columns for col in expected_cols):
            df = df.merge(cache_df[expected_cols], on=group_keys, how='left', suffixes=('', '_cache'))
            for col in feature_cols:
                cache_col = f'{col}_cache'
                if cache_col in df.columns:
                    df[col] = df[col].fillna(df[cache_col])
                    df.drop(columns=[cache_col], inplace=True)
            cached_groups = set(tuple(row) for row in cache_df[group_keys].to_numpy())

    for name, group in grouped:
        # Chronos features are only generated for horizon 1.
        if int(name[-1]) != 1:
            continue

        if name in cached_groups and group[feature_cols].notna().all().all():
            continue

        indices = group.index
        values = get_values(name)
        context = torch.tensor(values[-context_len:], dtype=torch.float32).unsqueeze(0)

        for h in horizons:
            with torch.no_grad():
                forecast = chronos.predict(context, prediction_length=h, num_samples=num_samples)
                mean_pred = forecast.mean(dim=1).mean(dim=1).item()
                df.loc[indices, f'chronos_pred_h{h}'] = mean_pred

    for col in feature_cols:
        df.loc[df[HORIZON_COL] != 1, col] = np.nan
        df[col] = df.groupby(group_keys)[col].ffill().bfill()

    df, kept_cols = sanitize_chronos_features(df, feature_cols)

    if cache_path and kept_cols:
        cache_out = df.groupby(group_keys).tail(1)[group_keys + kept_cols]
        cache_out.to_parquet(cache_path, index=False)

    return df, kept_cols

---
## 7️⃣ Time-Series Cross Validation

Standard K-Fold is **invalid** for time series because it leaks future data into training.  
We use `TimeSeriesSplit` which ensures:

```
Fold 1:  [Train: t0–t20]  →  [Val: t21–t30]
Fold 2:  [Train: t0–t30]  →  [Val: t31–t40]
Fold 3:  [Train: t0–t40]  →  [Val: t41–t50]
  ...expanding window...
```

### Why This Matters
- **No future leakage** — validation data always comes after training data
- **Expanding window** — later folds see more data, mimicking real-world deployment
- **LB stability** — temporal CV correlates better with leaderboard than random splits

### Additional Tricks
- **Target encoding** is computed **per fold** from training data only → prevents leakage
- **Multi-seed averaging** (3 seeds) reduces variance within each fold

---
## 8️⃣ Model Training — LightGBM + CatBoost

We train **two diverse base models** to maximize ensemble gain:

| Model | Strengths | Strategy |
|---|---|---|
| **LightGBM** | Fast training, leaf-wise growth, excellent on large data | 3-seed averaging per fold |
| **CatBoost** | Ordered boosting, robust to overfitting on categoricals | Complements LGB's bias |

> 🎯 **Diversity is key:** Using models with different learning algorithms reduces correlated errors, which is exactly what stacking exploits.

In [7]:
def remove_bad_features(df, features, target):
    good_features = []
    for col in features:
        if col not in df.columns:
            continue
        if df[col].isna().mean() > 0.5:
            continue
        if df[col].nunique(dropna=True) <= 1:
            continue
        corr = df[[col, target]].corr(numeric_only=True).iloc[0, 1]
        if pd.isna(corr) or abs(corr) < 0.01:
            continue
        good_features.append(col)
    return good_features


def get_features_for_horizon(df_h, base_features, chronos_cols, target=TARGET):
    current = [c for c in base_features if c in df_h.columns]
    if int(df_h[HORIZON_COL].iloc[0]) != 1:
        current = [c for c in current if c not in chronos_cols]
    return remove_bad_features(df_h, current, target)


def train_cv_with_target_encoding(df_h, base_features, chronos_cols, target=TARGET, n_splits=N_SPLITS):
    df_h = df_h.sort_values(TIME_COL).copy()
    original_index = df_h.index.to_numpy()
    df_h = df_h.reset_index(drop=True)
    horizon_value = int(df_h[HORIZON_COL].iloc[0])

    selected_features = get_features_for_horizon(df_h, base_features, chronos_cols, target=target)
    print(f"Horizon {horizon_value}: {len(selected_features)} filtered base features")

    unique_times = np.sort(df_h[TIME_COL].unique())
    if len(unique_times) <= (n_splits + 1):
        raise ValueError(f"Not enough unique time points for horizon {horizon_value}: {len(unique_times)}")

    tscv = TimeSeriesSplit(n_splits=n_splits)
    oof_lgb = np.full(len(df_h), np.nan)
    oof_cat = np.full(len(df_h), np.nan)

    lgb_models = []
    cat_models = []

    for fold, (train_time_idx, val_time_idx) in enumerate(tscv.split(unique_times)):
        train_times = unique_times[train_time_idx]
        val_times = unique_times[val_time_idx]

        train_df = df_h[df_h[TIME_COL].isin(train_times)].copy()
        val_df = df_h[df_h[TIME_COL].isin(val_times)].copy()

        # Target encoding from train fold only; unseen categories use fold global mean.
        for col in CAT_COLS:
            target_mean = train_df.groupby(col)[target].mean().to_dict()
            global_mean = train_df[target].mean()
            train_df[f'{col}_te'] = train_df[col].map(target_mean).fillna(global_mean)
            val_df[f'{col}_te'] = val_df[col].map(target_mean).fillna(global_mean)

        te_cols = [f'{col}_te' for col in CAT_COLS]
        current_features = selected_features + te_cols

        for col in current_features:
            if col not in train_df.columns:
                train_df[col] = -1
            if col not in val_df.columns:
                val_df[col] = -1

        X_train = train_df[current_features].fillna(-1)
        y_train = train_df[target]
        X_val = val_df[current_features].fillna(-1)
        y_val = val_df[target]

        lgb_params = {
            'objective': 'mae',
            'metric': 'mae',
            'learning_rate': 0.03,
            'max_depth': 7,
            'num_leaves': 63,
            'feature_fraction': 0.7,
            'bagging_fraction': 0.8,
            'bagging_freq': 1,
            'lambda_l1': 0.1,
            'lambda_l2': 0.1,
            'verbose': -1,
        }

        lgb_fold = np.zeros(len(val_df))
        for seed in SEEDS:
            lgb_params['seed'] = seed
            model = lgb.LGBMRegressor(**lgb_params, n_estimators=1500)
            model.fit(
                X_train,
                y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(stopping_rounds=150, verbose=False)],
            )
            lgb_fold += model.predict(X_val) / len(SEEDS)
            lgb_models.append(model)

        cat_model = CatBoostRegressor(
            iterations=1500,
            learning_rate=0.03,
            depth=7,
            loss_function='MAE',
            eval_metric='MAE',
            verbose=0,
        )
        cat_model.fit(X_train, y_train)
        cat_fold = cat_model.predict(X_val)
        cat_models.append(cat_model)

        oof_lgb[val_df.index.to_numpy()] = lgb_fold
        oof_cat[val_df.index.to_numpy()] = cat_fold

        mae_lgb = mean_absolute_error(y_val, lgb_fold)
        mae_cat = mean_absolute_error(y_val, cat_fold)
        mae_blend = mean_absolute_error(y_val, 0.5 * (lgb_fold + cat_fold))
        print(
            f"H={horizon_value} Fold {fold + 1} MAE | "
            f"LGB: {mae_lgb:.5f} | CAT: {mae_cat:.5f} | Blend: {mae_blend:.5f}"
        )

    return {
        'horizon': horizon_value,
        'original_index': original_index,
        'features': selected_features + [f'{col}_te' for col in CAT_COLS],
        'lgb_models': lgb_models,
        'cat_models': cat_models,
        'oof_lgb': oof_lgb,
        'oof_cat': oof_cat,
    }

---
## 9️⃣ Execute Training Pipeline

The pipeline runs in strict order:  
**Chronos features → Statistical features → CV training**

> Each step depends on the previous — Chronos features must exist before CV, and statistical features feed into the model.

In [8]:
# -- Step 1: Chronos Feature Extraction ---------------------------------
df, chronos_cols = generate_chronos_features_batch(
    df_raw,
    horizons=CHRONOS_HORIZONS,
    num_samples=CHRONOS_NUM_SAMPLES,
    cache_path=CHRONOS_CACHE_PATH,
)

# -- Step 2: Statistical Feature Engineering -----------------------------
df = build_statistical_features(df)

# -- Step 3: Define feature candidates -----------------------------------
exclude_cols = [TARGET] + GROUP_COLS + CAT_COLS + [TIME_COL, HORIZON_COL, ID_COL, 'weight']
base_features = [col for col in df.columns if col not in exclude_cols]

# -- Step 4: Horizon-wise training + stacking ----------------------------
print("\nStarting Horizon-Aware Cross-Validation Training...")

horizon_models = {}
df['oof_lgb'] = np.nan
df['oof_cat'] = np.nan
df['oof_final'] = np.nan

for h in SUPPORTED_HORIZONS:
    df_h = df[df[HORIZON_COL] == h].copy()
    if df_h.empty:
        print(f"Horizon {h}: no rows, skipping")
        continue

    print(f"\n========== Horizon {h} ==========")
    result = train_cv_with_target_encoding(
        df_h=df_h,
        base_features=base_features,
        chronos_cols=chronos_cols,
        target=TARGET,
        n_splits=N_SPLITS,
    )

    oof_stack = np.column_stack([result['oof_lgb'], result['oof_cat']])
    valid_idx = ~np.isnan(oof_stack).any(axis=1)

    ridge = Ridge(alpha=1.0)
    ridge.fit(oof_stack[valid_idx], df_h.loc[result['original_index'][valid_idx], TARGET])

    coef_abs = np.abs(ridge.coef_)
    use_fallback = bool(np.all(coef_abs < 1e-3))

    print(f"H={h} Ridge coefficients: {ridge.coef_}")
    print(f"H={h} Ridge intercept: {ridge.intercept_:.6f}")
    if use_fallback:
        print(f"H={h} Ridge collapsed -> using 0.5/0.5 fallback blend")

    oof_final = np.full(len(result['oof_lgb']), np.nan)
    if use_fallback:
        oof_final[valid_idx] = 0.5 * result['oof_lgb'][valid_idx] + 0.5 * result['oof_cat'][valid_idx]
    else:
        oof_final[valid_idx] = ridge.predict(oof_stack[valid_idx])

    df.loc[result['original_index'], 'oof_lgb'] = result['oof_lgb']
    df.loc[result['original_index'], 'oof_cat'] = result['oof_cat']
    df.loc[result['original_index'], 'oof_final'] = oof_final

    horizon_models[h] = {
        'features': result['features'],
        'lgb_models': result['lgb_models'],
        'cat_models': result['cat_models'],
        'ridge': ridge,
        'use_fallback': use_fallback,
    }

print("\nTraining complete for horizons:", sorted(horizon_models.keys()))

Extracting Chronos features...

Starting Horizon-Aware Cross-Validation Training...

========== Horizon 1 ==========
Horizon 1: 16 filtered base features
H=1 Fold 1 MAE | LGB: 1.23384 | CAT: 1.31930 | Blend: 1.24770
H=1 Fold 2 MAE | LGB: 0.73944 | CAT: 0.80196 | Blend: 0.75822
H=1 Fold 3 MAE | LGB: 1.52100 | CAT: 1.63232 | Blend: 1.55594
H=1 Fold 4 MAE | LGB: 1.24833 | CAT: 1.35238 | Blend: 1.28044
H=1 Fold 5 MAE | LGB: 1.20454 | CAT: 1.30677 | Blend: 1.23509
H=1 Ridge coefficients: [ 1.16928255 -0.08920408]
H=1 Ridge intercept: 0.065883

========== Horizon 3 ==========
Horizon 3: 33 filtered base features
H=3 Fold 1 MAE | LGB: 1.84218 | CAT: 2.02329 | Blend: 1.87499
H=3 Fold 2 MAE | LGB: 1.09279 | CAT: 1.19545 | Blend: 1.12100
H=3 Fold 3 MAE | LGB: 2.16065 | CAT: 2.36377 | Blend: 2.22484
H=3 Fold 4 MAE | LGB: 1.75138 | CAT: 1.96860 | Blend: 1.82595
H=3 Fold 5 MAE | LGB: 1.69977 | CAT: 1.93837 | Blend: 1.78075
H=3 Ridge coefficients: [0.92509616 0.13375362]
H=3 Ridge intercept: 0.02200

---
## 🔟 Horizon-Wise OOF Stacking

Stacking is now performed **separately for each horizon** (`1, 3, 10, 25`) to avoid cross-horizon bias.

### Key Fixes

1. Train LightGBM + CatBoost independently per horizon
2. Learn Ridge stacker per horizon using OOF predictions
3. Print Ridge coefficients for each horizon
4. If coefficients collapse near zero, auto-fallback to:

```python
final_pred = 0.5 * lgb_pred + 0.5 * cat_pred
```

This prevents the stacker from shrinking predictions toward constant near-zero values.

In [9]:
# -- OOF Diagnostics after horizon-wise stacking -------------------------
print("OOF diagnostics by horizon")
for h in sorted(horizon_models.keys()):
    mask = df[HORIZON_COL] == h
    pred_h = df.loc[mask, 'oof_final'].to_numpy()
    zero_rate = (np.isclose(pred_h, 0.0, atol=1e-12)).mean() if len(pred_h) else np.nan
    print(
        f"H={h} | count={len(pred_h)} | zero%={100 * zero_rate:.2f}% | "
        f"mean={np.nanmean(pred_h):.6f} | std={np.nanstd(pred_h):.6f}"
    )

OOF diagnostics by horizon
H=1 | count=1394653 | zero%=0.00% | mean=-0.071122 | std=8.591153
H=3 | count=1385816 | zero%=0.00% | mean=-0.203413 | std=16.007937
H=10 | count=1337236 | zero%=0.00% | mean=-0.641879 | std=27.489598
H=25 | count=1219709 | zero%=0.00% | mean=-1.497084 | std=45.663635


---
## 1️⃣1️⃣ Test Inference & Final Prediction

For the test set, we:
1. Apply **identical** feature engineering (Chronos + statistical)
2. Predict with **all trained models** (all folds × all seeds)
3. Average predictions across models
4. Pass through **Ridge stacker** for the final blend
5. **Clip** predictions to a sensible range `[0, 1000]`

> ⚠️ The test feature pipeline must be **exactly the same** as training — any mismatch causes silent bugs.

In [10]:
# -- Apply identical feature engineering as training ---------------------
test_df, test_chronos_cols = generate_chronos_features_batch(
    test_df,
    history_df=df_raw,
    horizons=CHRONOS_HORIZONS,
    num_samples=CHRONOS_NUM_SAMPLES,
    cache_path=CHRONOS_CACHE_PATH,
)
test_df = build_statistical_features(test_df, history_df=df_raw)

# -- Target encoding for test (fit on full train) ------------------------
for col in CAT_COLS:
    target_mean = df_raw.groupby(col)[TARGET].mean().to_dict()
    global_mean = df_raw[TARGET].mean()
    test_df[f'{col}_te'] = test_df[col].map(target_mean).fillna(global_mean)

# -- Sanity check: lag zeros in test -------------------------------------
lag_cols = [c for c in test_df.columns if c.startswith('lag_')]
if lag_cols:
    lag_zero_ratio = (test_df[lag_cols] == 0).mean().mean()
    print(f"Average zero ratio across lag features in test: {lag_zero_ratio:.2%}")

# -- Horizon-wise inference -----------------------------------------------
print("\nRunning horizon-aware inference on test set...")
test_df['pred_lgb'] = np.nan
test_df['pred_cat'] = np.nan
test_df['pred_final'] = np.nan

def ensure_features(frame, cols):
    for col in cols:
        if col not in frame.columns:
            frame[col] = -1
    return frame

for h, info in horizon_models.items():
    mask = test_df[HORIZON_COL] == h
    if not mask.any():
        continue

    feature_cols = info['features']
    test_df = ensure_features(test_df, feature_cols)
    X_test_h = test_df.loc[mask, feature_cols].fillna(-1)

    lgb_pred = np.zeros(mask.sum())
    for model in info['lgb_models']:
        lgb_pred += model.predict(X_test_h) / len(info['lgb_models'])

    cat_pred = np.zeros(mask.sum())
    for model in info['cat_models']:
        cat_pred += model.predict(X_test_h) / len(info['cat_models'])

    if info['use_fallback']:
        pred_h = 0.5 * lgb_pred + 0.5 * cat_pred
    else:
        stack_h = np.column_stack([lgb_pred, cat_pred])
        pred_h = info['ridge'].predict(stack_h)

    pred_h = np.clip(pred_h, 1e-6, None)

    test_df.loc[mask, 'pred_lgb'] = lgb_pred
    test_df.loc[mask, 'pred_cat'] = cat_pred
    test_df.loc[mask, 'pred_final'] = pred_h

# Fallback for unseen horizons in test
missing_mask = test_df['pred_final'].isna()
if missing_mask.any():
    fallback_value = max(df[TARGET].median(), 1e-6)
    print(f"Rows without trained horizon model: {missing_mask.sum()} -> filling with median={fallback_value:.6f}")
    test_df.loc[missing_mask, 'pred_final'] = fallback_value

test_preds = np.clip(test_df['pred_final'].to_numpy(), a_min=CLIP_MIN, a_max=CLIP_MAX)

Extracting Chronos features...
Average zero ratio across lag features in test: 0.00%

Running horizon-aware inference on test set...


---
## 1️⃣2️⃣ Prediction Diagnostics (Anti-Zero Collapse)

We explicitly audit prediction behavior before submission:

### Overall checks
- `% predictions == 0`
- `mean` / `median`

### Horizon checks
- `count`
- `% zero predictions`
- `mean` / `std`

If a horizon has a very high zero rate, we print a warning to flag likely feature/model issues.

---
## 1️⃣3️⃣ Submission

The submission file follows the standard Kaggle format:

| Column | Description |
|---|---|
| `id` | Row identifier from `test.parquet` |
| `prediction` | Final forecasted values |

File is saved directly to `/kaggle/working/` for easy download.

In [11]:
# -- Prediction Diagnostics ----------------------------------------------
overall_zero_rate = (np.isclose(test_preds, 0.0, atol=1e-12)).mean()
print("Overall Prediction Diagnostics")
print(f"Rows: {len(test_preds)}")
print(f"% predictions == 0: {100 * overall_zero_rate:.2f}%")
print(f"mean: {np.mean(test_preds):.6f}")
print(f"median: {np.median(test_preds):.6f}")

print("\nBy-Horizon Diagnostics")
for h in sorted(test_df[HORIZON_COL].dropna().unique()):
    mask = test_df[HORIZON_COL] == h
    pred_h = test_preds[mask.to_numpy()]
    if len(pred_h) == 0:
        continue
    zero_rate_h = (np.isclose(pred_h, 0.0, atol=1e-12)).mean()
    print(
        f"H={int(h)} | count={len(pred_h)} | zero%={100 * zero_rate_h:.2f}% | "
        f"mean={np.mean(pred_h):.6f} | std={np.std(pred_h):.6f}"
    )
    if zero_rate_h > 0.60:
        print(f"Warning: Horizon {int(h)} likely broken (high zero rate)")

# -- Create Submission File ----------------------------------------------
submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    PRED_COL: test_preds
})
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"\nSubmission saved to: {SUBMISSION_PATH}")
print(f"Shape: {submission.shape}")
print(f"Prediction range: [{submission[PRED_COL].min():.6f}, {submission[PRED_COL].max():.6f}]")
submission.head()

Overall Prediction Diagnostics
Rows: 1447107
% predictions == 0: 0.00%
mean: 0.354174
median: 0.000001

By-Horizon Diagnostics
H=1 | count=379617 | zero%=0.00% | mean=0.289185 | std=3.340921
H=3 | count=376558 | zero%=0.00% | mean=0.205149 | std=2.077869
H=10 | count=362057 | zero%=0.00% | mean=0.382205 | std=2.770348
H=25 | count=328875 | zero%=0.00% | mean=0.568962 | std=2.808718

Submission saved to: /kaggle/working/submission.csv
Shape: (1447107, 2)
Prediction range: [0.000001, 90.641617]


,id,prediction
0,10BAVIDU__07YQ9WA4__PZ9S1Z4V__1__4175,0.000001
1,10BAVIDU__07YQ9WA4__V8BKY1IV__1__4175,0.766840
2,10BAVIDU__07YQ9WA4__DPPUO5X2__1__4175,0.000001
3,10BAVIDU__07YQ9WA4__NQ58FVQM__1__4175,0.000001
4,10BAVIDU__07YQ9WA4__PHHHVYZI__1__4175,0.000001


---

### 📝 Notes

- **Before submitting:** Update all file paths to match your competition dataset
- **GPU:** Chronos-2 benefits significantly from GPU — ensure GPU accelerator is enabled
- **Tuning:** The LightGBM hyperparameters above are a strong baseline; consider Optuna for further gains
- **Extensions:** Add more Chronos horizons, additional lag windows, or interaction features

---
*Built for competition. Good luck! 🚀*